# 11 — Bronze: Receita Federal — Tabelas de Domínio

## Purpose
Ingests Receita Federal domain/lookup tables into the Bronze layer.
Processes 6 small reference files: Cnaes, Municipios, Naturezas, Qualificacoes, Paises, Motivos.

## Input
- `data/raw/receita_federal/{YYYY-MM}/{Table}.zip` — one ZIP per domain table
- All files < 1 MB — static reference data updated monthly with main snapshot

## Output
- `data/bronze/rf_{table}/_year_month={YYYY-MM}/data.parquet` — one Parquet per domain table
- 6 tables: rf_cnaes, rf_municipios, rf_naturezas, rf_qualificacoes, rf_paises, rf_motivos

## Steps
1. Imports and configuration
2. Schema definition
3. Process domain tables
4. Validation
5. Ops registration

## Notes
- No column headers — column names assigned from official RF layout (2 columns: code + description)
- Encoding: latin-1 with utf-8/cp1252 fallback
- All columns as STRING (ADR-002)
- No `_rescued_data` — domain tables have fixed 2-column layout, only drift = column count change
- Pandas used for ZIP/CSV reading (files < 1 MB — no memory pressure)
- Idempotent — existing partitions overwritten on re-run


## Step 1 — Imports and configuration

In [ ]:
import io
import json
import sys
import uuid
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd  # used only in read_zip_csv — small files, encoding fallback

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry
from pipeline import check_landing_gate

PROJECT_ROOT = get_project_root()

RF_ROOT     = PROJECT_ROOT / "data" / "raw" / "receita_federal"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
LOG_PATH    = PROJECT_ROOT / "db"  / "bootstrap_log.json"

check_landing_gate(PROJECT_ROOT)

available_months = sorted([d.name for d in RF_ROOT.iterdir() if d.is_dir()])
if not available_months:
    raise FileNotFoundError(
        f"No month directories found in {RF_ROOT}\n"
        "Run 01_bootstrap_receita_federal.py first."
    )

SAMPLE_MONTH = available_months[-1]
SAMPLE_DIR   = RF_ROOT / SAMPLE_MONTH

SOURCE_ID  = "receita_federal"
BATCH_ID   = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).isoformat()

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"SAMPLE_MONTH : {SAMPLE_MONTH}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"STARTED_AT   : {STARTED_AT}")


## Step 2 — Schema definition

In [ ]:
# ---------------------------------------------------------------------------
# Domain table schemas — 2 columns each per official RF layout
# Format: code (INTEGER in source, stored as STRING) + description (VARCHAR)
# ---------------------------------------------------------------------------
DOMAIN_TABLES = {
    "rf_cnaes"        : {"zip": "Cnaes.zip",        "cols": ["cnae_codigo",         "cnae_descricao"]},
    "rf_municipios"   : {"zip": "Municipios.zip",    "cols": ["municipio_codigo",    "municipio_descricao"]},
    "rf_naturezas"    : {"zip": "Naturezas.zip",     "cols": ["natureza_codigo",     "natureza_descricao"]},
    "rf_qualificacoes": {"zip": "Qualificacoes.zip", "cols": ["qualificacao_codigo", "qualificacao_descricao"]},
    "rf_paises"       : {"zip": "Paises.zip",        "cols": ["pais_codigo",         "pais_descricao"]},
    "rf_motivos"      : {"zip": "Motivos.zip",       "cols": ["motivo_codigo",       "motivo_descricao"]},
}

TECHNICAL_COLUMNS = ["_source_id", "_batch_id", "_ingested_at", "_source_file", "_year_month"]

print("Domain tables defined:")
for table, meta in DOMAIN_TABLES.items():
    total_cols = len(meta["cols"]) + len(TECHNICAL_COLUMNS)
    print(f"  {table:<22} {meta['zip']:<22} source_cols={len(meta['cols'])} total={total_cols}")


## Step 3 — Process domain tables

In [ ]:
def read_zip_csv(zip_path: Path, encoding: str = "latin-1") -> pd.DataFrame:
    """
    Read a domain table CSV from a ZIP file into a pandas DataFrame.

    Domain files are < 1 MB — pandas in-memory reading is appropriate here.
    Encoding fallback covers RF files with inconsistent encoding.

    Parameters
    ----------
    zip_path : path to the .zip file
    encoding : primary encoding (default: latin-1)

    Returns
    -------
    pd.DataFrame — all columns as string, no headers
    """
    with zipfile.ZipFile(zip_path) as zf:
        csv_bytes = zf.read(zf.namelist()[0])

    for enc in [encoding, "utf-8", "cp1252"]:
        try:
            return pd.read_csv(
                io.BytesIO(csv_bytes),
                sep=";", header=None, dtype=str,
                encoding=enc, on_bad_lines="skip",
            )
        except (UnicodeDecodeError, Exception):
            continue

    raise ValueError(f"Could not decode {zip_path.name} with any known encoding")


def process_domain(table_name: str, meta: dict, con) -> dict:
    """
    Read one domain table ZIP, apply Bronze schema, and write Parquet.

    Column count is validated before writing — mismatch returns ERROR status.
    No `_rescued_data` — domain tables have fixed 2-column layout.

    Parameters
    ----------
    table_name : output table name (e.g. "rf_cnaes")
    meta       : dict with keys "zip" (filename) and "cols" (column names)
    con        : DuckDB connection

    Returns
    -------
    dict with keys: table, status, records, file, error
    """
    zip_path = SAMPLE_DIR / meta["zip"]

    if not zip_path.exists():
        return {"table": table_name, "status": "MISSING", "records": 0,
                "file": None, "error": f"File not found: {zip_path}"}

    partition_dir = BRONZE_ROOT / table_name / f"_year_month={SAMPLE_MONTH}"
    partition_dir.mkdir(parents=True, exist_ok=True)
    output_file = partition_dir / "data.parquet"
    output_path = to_sql_path(output_file)
    source_sql  = to_sql_path(zip_path)

    try:
        df = read_zip_csv(zip_path)

        # Column count validation — only drift possible for 2-column tables
        expected_cols = len(meta["cols"])
        actual_cols   = len(df.columns)
        if actual_cols != expected_cols:
            return {"table": table_name, "status": "ERROR", "records": 0,
                    "file": None,
                    "error": f"Column count mismatch: expected {expected_cols}, got {actual_cols}"}

        # Build column aliases: "0" AS col0, "1" AS col1
        col_aliases = ", ".join(f'"{i}" AS {c}' for i, c in enumerate(meta["cols"]))

        con.execute(f"""
            COPY (
                SELECT
                    {col_aliases},
                    '{SOURCE_ID}'    AS _source_id,
                    '{BATCH_ID}'     AS _batch_id,
                    CURRENT_TIMESTAMP AS _ingested_at,
                    '{source_sql}'   AS _source_file,
                    '{SAMPLE_MONTH}' AS _year_month
                FROM df
            ) TO '{output_path}'
            (FORMAT PARQUET, COMPRESSION SNAPPY)
        """)

        records = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
        ).fetchone()[0]

        return {"table": table_name, "status": "SUCCESS", "records": records,
                "file": str(output_file), "error": None}

    except Exception as e:
        return {"table": table_name, "status": "ERROR", "records": 0,
                "file": None, "error": str(e)}


# ---------------------------------------------------------------------------
# Execute — process all 6 domain tables
# ---------------------------------------------------------------------------
results = []
con = get_connection()  # in-memory — Parquet I/O does not require persistent DB

for table_name, meta in DOMAIN_TABLES.items():
    res = process_domain(table_name, meta, con)
    results.append(res)
    icon = "OK " if res["status"] == "SUCCESS" else "ERR"
    print(f"  [{icon}] {table_name:<22} {res['status']:<8} {res['records']:>6,} rows")

con.close()

error_count   = sum(1 for r in results if r["status"] == "ERROR")
missing_count = sum(1 for r in results if r["status"] == "MISSING")
total_records = sum(r["records"] for r in results)

print()
print(f"Tables OK  : {len(results) - error_count - missing_count}/{len(results)}")
print(f"Errors     : {error_count}")
print(f"Missing    : {missing_count}")
print(f"Total rows : {total_records:,}")


## Step 4 — Validation

In [ ]:
suite   = CheckSuite("bronze_rf_dominios")
con_val = get_connection()

# Check 1 — no processing errors or missing files
suite.add(
    "No processing errors or missing files",
    error_count == 0 and missing_count == 0,
    f"errors={error_count}, missing={missing_count}",
)

# Check 2 — all 6 Parquet files exist
missing_files = [r["table"] for r in results
    if r["status"] == "SUCCESS" and not Path(r["file"]).exists()]
suite.add(
    "All 6 Parquet files exist",
    not missing_files,
    f"missing: {missing_files}" if missing_files else "all present",
)

# Check 3 — no empty tables
empty_tables = [r["table"] for r in results if r["status"] == "SUCCESS" and r["records"] == 0]
suite.add(
    "No empty tables",
    not empty_tables,
    f"empty: {empty_tables}" if empty_tables else "all non-empty",
)

# Check 4 — expected record count ranges per table
EXPECTED_RANGES = {
    "rf_cnaes"        : (1_000,  2_000),
    "rf_municipios"   : (5_000,  6_000),
    "rf_naturezas"    : (80,       200),
    "rf_qualificacoes": (60,       100),
    "rf_paises"       : (250,       400),
    "rf_motivos"      : (50,        100),
}
out_of_range = []
for r in results:
    if r["status"] == "SUCCESS" and r["table"] in EXPECTED_RANGES:
        lo, hi = EXPECTED_RANGES[r["table"]]
        if not (lo <= r["records"] <= hi):
            out_of_range.append(f"{r['table']}={r['records']:,} (expected {lo:,}-{hi:,})")
suite.add(
    "Record counts within expected ranges",
    not out_of_range,
    f"out of range: {out_of_range}" if out_of_range else "all within range",
)

# Check 5 — technical columns present in all tables
missing_tech = []
for r in results:
    if r["status"] == "SUCCESS":
        cols = {row[0] for row in con_val.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{to_sql_path(r['file'])}') LIMIT 0"
        ).fetchall()}
        for tc in TECHNICAL_COLUMNS:
            if tc not in cols:
                missing_tech.append(f"{r['table']}.{tc}")
suite.add(
    "Technical columns present in all tables",
    not missing_tech,
    f"missing: {missing_tech}" if missing_tech else "all present",
)

# Check 6 — all code columns are STRING (not INTEGER)
non_varchar = []
for r in results:
    if r["status"] == "SUCCESS":
        table_meta = DOMAIN_TABLES[r["table"]]
        code_col   = table_meta["cols"][0]  # always the code column
        col_types  = {row[0]: row[1] for row in con_val.execute(
            f"DESCRIBE SELECT * FROM read_parquet('{to_sql_path(r['file'])}') LIMIT 0"
        ).fetchall()}
        if col_types.get(code_col) != "VARCHAR":
            non_varchar.append(f"{r['table']}.{code_col}={col_types.get(code_col)}")
suite.add(
    "All code columns are VARCHAR (not INTEGER)",
    not non_varchar,
    f"non-varchar: {non_varchar}" if non_varchar else "all VARCHAR",
)

# Check 7 — record count summary
print("  Record counts:")
for r in results:
    print(f"    {r['table']:<22} {r['records']:>6,} rows")
print()

con_val.close()  # close before potential raise
suite.report()
suite.assert_all_pass()


## Step 5 — Ops registration

In [ ]:
files_written = sum(1 for r in results if r["status"] == "SUCCESS")
bytes_written = sum(
    Path(r["file"]).stat().st_size
    for r in results if r["status"] == "SUCCESS" and r["file"]
)
errors_detail = (
    "; ".join(f"{r['table']}: {r['error'][:60]}" for r in results if r["status"] == "ERROR")
    if error_count > 0 else None
)

entry = make_entry(
    source_id     = SOURCE_ID,
    period        = SAMPLE_MONTH,
    status        = "SUCCESS" if error_count == 0 else "PARTIAL",
    records       = total_records,
    bytes_written = bytes_written,
    started_at    = STARTED_AT,
    finished_at   = datetime.now(timezone.utc).isoformat(),
    error_message = errors_detail,
    batch_id         = BATCH_ID,
    layer            = "bronze",
    object           = "rf_dominios",
    files            = files_written,
    has_rescued_data = False,
    drift_months     = 0,
)

append_log(LOG_PATH, entry)

print(f"Batch registered   : {BATCH_ID}")
print(f"Status             : {entry['status']}")
print(f"Tables             : {files_written}/6")
print(f"Records            : {total_records:,}")
print(f"Bytes written      : {bytes_written / 1024:.1f} KB")
print(f"Log                : {LOG_PATH}")
